# ASF-S — CIFAR-100 / Conv2Net on Kaggle

Runs `cifar100/` **unchanged** apart from model selection, which the repo does by
commenting code in and out. Every algorithm line is untouched.

**Before you start:** Settings → *Accelerator* = **GPU T4 x2** (or P100), and
*Internet* = **On** (needed for `pip install ptflops` and the CIFAR-100 download).

**Run it with `Save & Run All (Commit)`, not interactively.** The full Conv2Net
pipeline is ~9-10 h and an interactive session will not survive that; a committed
run gets the full 12 h budget and keeps everything in `/kaggle/working`.

| Step | Script | Est. on T4 |
|---|---|---|
| 1 | `train_cifar100.py` | 20-30 min |
| 2 | `activation_ext_cifar100.py` | 45-70 min |
| 3 | `cosine_sim_ext_cifar100.py` | 15-25 min |
| 4 | `pg_ext_cifar100.py` | < 1 min |
| 5 | `hrank_cifar100.py` | ~30 min |
| 6 | `pg_pruning_cifar100.py` | **~7 h** (81-combo grid) |

In [ ]:
!nvidia-smi
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('python', sys.version)

## 1. Clone the repo

Cloning into `/kaggle/working` means `config.py`'s `BASE_DIR` (`HERE/pg_project_output`)
already lands in the persisted output directory — **no path edit needed**.

In [ ]:
%cd /kaggle/working
!rm -rf ASF-S
!git clone --depth 1 https://github.com/NeuroPulse-Co/ASF-S.git
!pip -q install ptflops
%cd /kaggle/working/ASF-S
!git log --oneline -1

## 2. Keep activations out of the 20 GB output quota

Conv2Net dumps ~20,000 `.pt` files (~9.4 GB): every layer in `LAYERS['conv2net']`
for up to 100 correct + 100 incorrect samples per class. `/kaggle/working` only
persists 20 GB, and committing 9.4 GB of intermediates is slow and pointless —
only `pg_data/` and `results/` are needed downstream.

Symlinking `activations/` to `/kaggle/temp` (scratch, not saved) keeps the code
untouched: `config.py` calls `os.makedirs(..., exist_ok=True)`, which is happy
with an existing symlink.

In [ ]:
!mkdir -p /kaggle/temp/activations
!mkdir -p /kaggle/working/ASF-S/cifar100/pg_project_output
!ln -sfn /kaggle/temp/activations /kaggle/working/ASF-S/cifar100/pg_project_output/activations
!ls -la /kaggle/working/ASF-S/cifar100/pg_project_output/

## 3. Select conv2net

The repo selects the model by commenting blocks in and out across five files.
This cell performs exactly the edits you would make by hand, using **exact block
replacement with an assert on every one** — if the repo changes, this fails loudly
instead of silently editing the wrong thing. Nothing else in the files is touched.

(`hrank_cifar100.py` is already set to `conv2net`, so it needs no edit.)

In [ ]:
import pathlib

def patch(relpath, old, new):
    p = pathlib.Path(relpath)
    s = p.read_text()
    assert s.count(old) == 1, f'{relpath}: expected exactly 1 match, got {s.count(old)}'
    p.write_text(s.replace(old, new))
    print(f'patched {relpath}')

# ---- train_cifar100.py : uncomment Conv2Net, comment VGG16 ----
patch('cifar100/train_cifar100.py',
      '\n'.join([
          '# m   = Conv2Net(NUM_CLASSES).to(device)',
          '# opt = optim.Adam(m.parameters(), lr=2e-4, weight_decay=5e-4)',
          '# sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.1)',
          '# train(m, opt, sch, num_epochs=200, patience=15, model_name="conv2net")',
      ]),
      '\n'.join([
          'm   = Conv2Net(NUM_CLASSES).to(device)',
          'opt = optim.Adam(m.parameters(), lr=2e-4, weight_decay=5e-4)',
          'sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.1)',
          'train(m, opt, sch, num_epochs=200, patience=15, model_name="conv2net")',
      ]))

patch('cifar100/train_cifar100.py',
      '\n'.join([
          'm   = VGG16CIFAR(NUM_CLASSES).to(device)',
          'opt = optim.SGD(m.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)',
          '# cosine schedule steps every epoch; pass None to train() and step manually',
          'sch_cos = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)',
          'train(m, opt, scheduler=None, num_epochs=200, patience=20, model_name="vgg16")',
      ]),
      '\n'.join([
          '# m   = VGG16CIFAR(NUM_CLASSES).to(device)',
          '# opt = optim.SGD(m.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)',
          '# cosine schedule steps every epoch; pass None to train() and step manually',
          '# sch_cos = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)',
          '# train(m, opt, scheduler=None, num_epochs=200, patience=20, model_name="vgg16")',
      ]))

# ---- activation_ext_cifar100.py ----
patch('cifar100/activation_ext_cifar100.py',
      '    # "conv2net": (Conv2Net(NUM_CLASSES),   LAYERS["conv2net"], "conv2net_best.pth"),',
      '    "conv2net": (Conv2Net(NUM_CLASSES),   LAYERS["conv2net"], "conv2net_best.pth"),')
patch('cifar100/activation_ext_cifar100.py',
      '    "vgg16":    (VGG16CIFAR(NUM_CLASSES), LAYERS["vgg16"],    "vgg16_best.pth"),',
      '    # "vgg16":    (VGG16CIFAR(NUM_CLASSES), LAYERS["vgg16"],    "vgg16_best.pth"),')

# ---- cosine_sim_ext_cifar100.py ----
patch('cifar100/cosine_sim_ext_cifar100.py',
      '    # "conv2net": LAYERS["conv2net"],',
      '    "conv2net": LAYERS["conv2net"],')
patch('cifar100/cosine_sim_ext_cifar100.py',
      '    "vgg16":    LAYERS["vgg16"],',
      '    # "vgg16":    LAYERS["vgg16"],')

# ---- pg_ext_cifar100.py ----
patch('cifar100/pg_ext_cifar100.py',
      '    # "conv2net": [PG_FC_LAYER["conv2net"]],',
      '    "conv2net": [PG_FC_LAYER["conv2net"]],')
patch('cifar100/pg_ext_cifar100.py',
      '    "vgg16":    [PG_FC_LAYER["vgg16"]],',
      '    # "vgg16":    [PG_FC_LAYER["vgg16"]],')

# ---- pg_pruning_cifar100.py ----
patch('cifar100/pg_pruning_cifar100.py',
      'for mn in ["vgg16"]:',
      'for mn in ["conv2net"]:')

!git diff --stat

### Optional: fewer DataLoader workers

The scripts hardcode `num_workers=4` with `persistent_workers=True`. Kaggle gives
4 vCPUs, so 4 is borderline; drop to 2 if you see loader stalls. **Skip this cell**
to keep the code fully as-is.

In [ ]:
# !sed -i 's/num_workers=4/num_workers=2/g' cifar100/*.py && git diff --stat

## Step 1 — train

Conv2Net, Adam lr 2e-4, wd 5e-4, `ReduceLROnPlateau`, 200 epochs max with
early-stop patience 15. Expect ~40-45% val accuracy on CIFAR-100.

In [ ]:
%cd /kaggle/working/ASF-S/cifar100
!python train_cifar100.py

## Step 2 — activations

**This is the slowest non-pruning step, and most of it is avoidable.**
`extract()` rebuilds the class index with
`[i for i, (_, lbl) in enumerate(dataset) if lbl == class_id]` once per class, so it
decodes and transforms the entire 100k-sample subset 100 times — 10M image decodes
on 4 vCPUs. `datasets.CIFAR100` exposes `.targets`, which would make it one line.
Left as-is here; see the note at the bottom of this notebook.

In [ ]:
!python activation_ext_cifar100.py

## Step 3 — affinity matrices

Reads all ~9.4 GB back to build per-class (50x50) sample-affinity matrices.
These feed the AFC figures, **not** the pruner — safe to skip if you only want
the pruning numbers.

In [ ]:
!python cosine_sim_ext_cifar100.py

## Step 4 — PG1

Seconds, because `process()` reads one sample per (class, split).

In [ ]:
!python pg_ext_cifar100.py

## Step 5 — HRank baseline

Already set to `conv2net`. Six ratios x up to 200 FC fine-tune epochs.

In [ ]:
!python hrank_cifar100.py

## Step 6 — PG / ASF-S pruning

**~7 hours.** Conv2Net takes the 2-D threshold grid
(`itertools.product(THRESHOLDS, repeat=2)` = **81** combinations), each with up to
200 FC fine-tune epochs at patience 10.

Results append to `pg_conv2net_cifar100.csv` as they go, so a killed session keeps
whatever finished — but there is no skip-completed logic, so a re-run restarts at
combo 1 and repeats work.

For a first pass, uncomment the `sed` below to shrink `THRESHOLDS` to 3 values
(9 combos, ~45 min). Note this gives the 9 pairs drawn from {0.3, 0.5, 0.7} — it is
a coarser grid, not a shard of the 81.

In [ ]:
# !sed -i 's/^THRESHOLDS = .*/THRESHOLDS = [0.3, 0.5, 0.7]/' pg_pruning_cifar100.py
!grep -n '^THRESHOLDS' pg_pruning_cifar100.py
!python pg_pruning_cifar100.py

## Results

In [ ]:
import pandas as pd, glob, os
for f in sorted(glob.glob('pg_project_output/results/*conv2net*.csv')):
    print('='*70); print(f)
    try:
        print(pd.read_csv(f).to_string())
    except Exception as e:
        print('could not parse:', e)

## Keep the checkpoint for the next session

`pg_project_output/trained_models/` and `results/` persist in the commit output.
To chain sessions, add this notebook's output as a **Dataset** input to the next
one and copy `conv2net_best.pth` in, so Step 1 can be skipped.

---

## Running on Colab instead

Same cells, four differences:

1. Clone to `/content/ASF-S` — ephemeral, so mount Drive and copy
   `pg_project_output/{trained_models,results}` there at the end. Do **not** put
   `activations/` on Drive; 20,000 small files over Drive I/O is unusably slow.
2. Replace the `/kaggle/temp` symlink with `/content/activations`.
3. Colab has 2 vCPUs — run the `num_workers=2` cell.
4. No 12 h background execution on free Colab; it drops on idle. Expect to do
   Step 6 across several sessions, or use Kaggle for it.

---

## Known issues in this pipeline

Things to be aware of when reading the output, none of them patched here:

- **PG1 uses one sample per (class, split).** `pg_ext_cifar100.py:108` breaks after
  the first match, so Eq. 1's activation matrix has S=1 and the Eq. 2 cosine matrix
  is +-1 everywhere. Both this and `min_scripts/pg_ext_2.py:83` do it.
- **Threshold is not a sparsity ratio.** PG prunes on `importance > threshold`,
  the baselines prune to an exact ratio. Comparing PG at 0.7 against a baseline at
  0.7 is not like-for-like; match on *achieved* sparsity instead.
- **`svd_conv2net_cifar100.csv` has a corrupted header** (header and first data row
  concatenated), so pandas misparses it.
- The root `README.md` is stale; `cifar100/README.md` is the accurate one.